# 04 — Hybrid Retrieval

**Repo:** ai-learning-notes | **Folder:** rag-and-retrieval  
**Built alongside:** local-rag-llm (github.com/RT91-data/local-rag-llm)

---

## What this notebook covers

- Why neither dense nor sparse alone is sufficient
- Score fusion — the naive approach and why it breaks
- Reciprocal Rank Fusion (RRF) — rank-based fusion that avoids score incompatibility
- Weighted score fusion — what LangChain EnsembleRetriever actually does
- The 60/40 weighting in local-rag-llm — what it means and how to tune it
- Junk chunk filtering — why retrieval quality isn't just about the retriever
- The full retrieval pipeline from query to final candidates

**Pure Python only — no external dependencies.**

---
## 1. The Core Problem With Combining Two Retrievers

Dense retrieval returns similarity scores: `0.0 to 1.0` (cosine) or `0 to ∞` (L2 distance).  
BM25 returns relevance scores: `0 to ~15` depending on term frequency and corpus size.

**These scores are on completely different scales.**  
You cannot simply add them.

A BM25 score of 3.5 might be excellent.  
A cosine similarity of 0.85 might be excellent.  
But `3.5 + 0.85 = 4.35` and `0.5 + 0.85 = 1.35` — the BM25 score dominates the sum regardless of quality.

Two solutions exist:
1. **Normalise scores** before combining (fragile — depends on corpus statistics)
2. **Use ranks instead of scores** — Reciprocal Rank Fusion (robust, scale-independent)

In [ ]:
import math

# Demonstrating the score incompatibility problem

# Simulated results from two retrievers for the same query
# Dense scores: cosine similarity 0-1
# BM25 scores: raw BM25 relevance (different scale entirely)

dense_results = [
    ("ZAA chunk",       0.91),
    ("JIT chunk",       0.87),
    ("SPIFFE chunk",    0.72),
    ("Sandbox chunk",   0.68),
]

bm25_results = [
    ("SPIFFE chunk",    4.82),  # exact keyword match
    ("JIT chunk",       3.21),
    ("ZAA chunk",       2.15),
    ("Confused Deputy", 1.94),
]

print("SCORE INCOMPATIBILITY PROBLEM")
print("=" * 55)
print(f"\nDense retriever scores (cosine, 0-1 range):")
for name, score in dense_results:
    print(f"  {name:<20} {score:.3f}")

print(f"\nBM25 scores (raw relevance, 0-5+ range):")
for name, score in bm25_results:
    print(f"  {name:<20} {score:.3f}")

# Naive sum — BM25 scale dominates
all_chunks = set(n for n,_ in dense_results) | set(n for n,_ in bm25_results)
dense_dict = dict(dense_results)
bm25_dict  = dict(bm25_results)

print(f"\nNaive addition (WRONG — BM25 dominates):")
naive_scores = []
for chunk in all_chunks:
    d = dense_dict.get(chunk, 0)
    b = bm25_dict.get(chunk, 0)
    naive_scores.append((chunk, d + b, d, b))
naive_scores.sort(key=lambda x: x[1], reverse=True)
print(f"  {'Chunk':<20} {'Dense':>7} {'BM25':>7} {'Sum':>7}")
for name, total, d, b in naive_scores:
    print(f"  {name:<20} {d:>7.3f} {b:>7.3f} {total:>7.3f}")
print("  Problem: BM25 scores dwarf dense scores — ranking is BM25-dominated.")

---
## 2. Reciprocal Rank Fusion (RRF)

RRF solves the scale problem by discarding scores entirely and using only **ranks**.

**The formula:**
```
RRF_score(chunk) = Σ  1 / (k + rank_in_retriever_i)
                  i
```
where `k` is a constant (typically 60) that dampens the advantage of top-ranked items.

**Why k=60?**  
Empirically, k=60 was found in the original RRF paper to give the best fusion performance across many IR benchmarks. It prevents rank 1 from having an overwhelming advantage over rank 2.

**Properties of RRF:**
- Completely scale-independent — a BM25 score of 8.5 and a cosine of 0.95 are both just "rank 1"
- A chunk ranked #1 by both retrievers gets the highest combined score
- A chunk ranked #1 by one and missing from the other still scores reasonably
- Robust to one retriever having a very different score distribution

In [ ]:
def reciprocal_rank_fusion(ranked_lists, k=60):
    """
    Combine multiple ranked lists using RRF.
    ranked_lists: list of lists, each containing chunk names in rank order
    Returns: list of (chunk, rrf_score) sorted by score descending
    """
    scores = {}
    for ranked_list in ranked_lists:
        for rank, chunk in enumerate(ranked_list, start=1):
            scores[chunk] = scores.get(chunk, 0) + 1 / (k + rank)
    return sorted(scores.items(), key=lambda x: x[1], reverse=True)


# Same results as before — but now using only ranks
dense_ranked = [name for name, _ in dense_results]   # [ZAA, JIT, SPIFFE, Sandbox]
bm25_ranked  = [name for name, _ in bm25_results]    # [SPIFFE, JIT, ZAA, Confused Deputy]

rrf_results = reciprocal_rank_fusion([dense_ranked, bm25_ranked], k=60)

print("RECIPROCAL RANK FUSION")
print("=" * 60)
print(f"\nDense ranking:  {dense_ranked}")
print(f"BM25 ranking:   {bm25_ranked}")
print()
print(f"{'Rank':<6} {'RRF Score':>10}  Chunk  (Dense rank | BM25 rank)")
print("-" * 60)
for rank, (chunk, score) in enumerate(rrf_results, 1):
    d_rank = dense_ranked.index(chunk) + 1 if chunk in dense_ranked else "—"
    b_rank = bm25_ranked.index(chunk)  + 1 if chunk in bm25_ranked  else "—"
    print(f"{rank:<6} {score:>10.6f}  {chunk:<20} Dense#{d_rank} | BM25#{b_rank}")

print()
print("JIT chunk: Dense#2, BM25#2 → consistently ranked → rises to top.")
print("SPIFFE: Dense#3, BM25#1 → strong BM25 signal lifts it despite weaker dense.")
print("Scale of original scores is completely irrelevant — only rank matters.")

---
## 3. Weighted Score Fusion — What LangChain Actually Does

LangChain's `EnsembleRetriever` does **not** use RRF internally.  
It uses **weighted Reciprocal Rank Fusion** — still rank-based, but with per-retriever weights.

```
weighted_RRF_score(chunk) = Σ  weight_i / (k + rank_in_retriever_i)
                            i
```

With `weights=[0.4, 0.6]` for `[bm25, faiss]`:
- A chunk at rank 1 in BM25 contributes `0.4 / (60 + 1) = 0.00656`
- A chunk at rank 1 in FAISS contributes `0.6 / (60 + 1) = 0.00984`
- FAISS rank 1 is worth 1.5× more than BM25 rank 1

**The 60/40 weighting in local-rag-llm:**  
FAISS gets 60% because semantic matching handles the majority of document Q&A queries.  
BM25 gets 40% because exact-match is critical for IDs, codes, and technical terms.  
This is a reasonable default — but it should be validated with RAGAS evaluation  
across the specific document corpus you're working with.

In [ ]:
def weighted_rrf(ranked_lists, weights, k=60):
    """
    Weighted RRF — what LangChain EnsembleRetriever implements.
    ranked_lists: list of lists (one per retriever)
    weights: list of floats (one per retriever, should sum to 1.0)
    """
    assert len(ranked_lists) == len(weights)
    scores = {}
    for ranked_list, weight in zip(ranked_lists, weights):
        for rank, chunk in enumerate(ranked_list, start=1):
            scores[chunk] = scores.get(chunk, 0) + weight / (k + rank)
    return sorted(scores.items(), key=lambda x: x[1], reverse=True)


# Compare equal weights vs local-rag-llm 40/60 weighting
configs = [
    ("Equal (50/50)",           [0.5, 0.5]),
    ("BM25-heavy (70/30)",      [0.7, 0.3]),
    ("FAISS-heavy 60/40",       [0.4, 0.6]),  # local-rag-llm: [bm25, faiss]
    ("Dense-only (0/100)",      [0.0, 1.0]),
]

print("WEIGHTING COMPARISON — how weights shift final ranking")
print("=" * 65)
print(f"BM25 ranking:  {bm25_ranked}")
print(f"Dense ranking: {dense_ranked}")
print()

for label, weights in configs:
    results = weighted_rrf([bm25_ranked, dense_ranked], weights)
    ranking = [chunk for chunk, _ in results]
    print(f"{label:<25} → {ranking}")

print()
print("Key observation:")
print("FAISS-heavy (60/40) still surfaces SPIFFE chunk high despite its BM25 advantage,")
print("because BM25#1 + Dense#3 combined still beats Dense#1-only or BM25#1-only.")
print("Hybrid consistently outperforms either retriever alone.")

---
## 4. The Full Retrieval Pipeline in local-rag-llm

The EnsembleRetriever is just one stage. The full pipeline has more steps:

```
Query
  ↓
Query rewriting (Haiku — expand follow-ups)
  ↓
FAISS similarity_search_with_score (k=16) + threshold filter (≥0.3)
  ↓
BM25 retrieval (k=4)
  ↓
Merge + deduplicate (content hash)
  ↓
Junk chunk filter (remove TOC, header-only chunks)
  ↓
CrossEncoder rerank → top 5
  ↓
Final chunks → Claude
```

**Important:** local-rag-llm does NOT use EnsembleRetriever for the final merge.  
It runs FAISS and BM25 separately, then merges the results manually.  
This gives more control: the FAISS results go through the similarity threshold filter  
before merging, which EnsembleRetriever would bypass.

In [ ]:
# Simulating the full merge + dedup + junk filter pipeline

def is_junk_chunk(content):
    """Filter TOC and header-only chunks — from local-rag-llm evaluate_rag.py"""
    lower = content.lower()
    if "table of contents" in lower:
        return True
    lines = [l.strip() for l in content.split("\n") if l.strip()]
    if len(lines) >= 6:
        short = sum(1 for l in lines if len(l) < 60)
        if short / len(lines) > 0.70 and len(content) < 1500:
            return True
    return False


# Simulated retrieval results
faiss_docs = [
    {"content": "Zero Ambient Authority means agents must not inherit full admin privileges.",  "sim": 0.88, "source": "faiss"},
    {"content": "JIT downscoping scopes credentials to exact data sources needed per task.",   "sim": 0.82, "source": "faiss"},
    {"content": "Table of contents\nIntroduction...6\nSecurity...7\nPillar 1...9\nPillar 2...10","sim": 0.79, "source": "faiss"},  # junk
    {"content": "SPIFFE IDs provide cryptographic identity for every individual agent.",       "sim": 0.71, "source": "faiss"},
    {"content": "JIT downscoping scopes credentials to exact data sources needed per task.",   "sim": 0.68, "source": "faiss"},  # duplicate
]

bm25_docs = [
    {"content": "SPIFFE IDs provide cryptographic identity for every individual agent.",       "sim": None, "source": "bm25"},  # already in faiss
    {"content": "The Confused Deputy problem: prompt injection tricks over-privileged agents.","sim": None, "source": "bm25"},
    {"content": "Agents must authenticate using a dedicated agentic identity, not delegated.", "sim": None, "source": "bm25"},
]

print("FULL MERGE PIPELINE")
print("=" * 65)

# Step 1: Threshold filter on FAISS (keep sim >= 0.3)
THRESHOLD = 0.3
faiss_filtered = [d for d in faiss_docs if d["sim"] >= THRESHOLD]
print(f"Step 1 — FAISS threshold filter (>={THRESHOLD}):")
print(f"  In: {len(faiss_docs)} chunks, Out: {len(faiss_filtered)} chunks")

# Step 2: Merge FAISS + BM25
all_candidates = faiss_filtered + bm25_docs
print(f"\nStep 2 — Merge FAISS + BM25:")
print(f"  Total before dedup: {len(all_candidates)} chunks")

# Step 3: Deduplicate by content prefix
seen = set()
merged = []
for doc in all_candidates:
    key = doc["content"][:80]
    if key not in seen:
        seen.add(key)
        merged.append(doc)
print(f"  After dedup: {len(merged)} chunks")

# Step 4: Junk filter
clean = [d for d in merged if not is_junk_chunk(d["content"])]
junk  = [d for d in merged if is_junk_chunk(d["content"])]
print(f"\nStep 3 — Junk filter:")
for j in junk:
    print(f"  REMOVED: '{j['content'][:60]}...'")
print(f"  Clean chunks remaining: {len(clean)}")

print(f"\nFinal candidates for CrossEncoder reranking:")
for i, doc in enumerate(clean):
    src = f"[{doc['source']}]"
    sim = f"sim={doc['sim']:.2f}" if doc['sim'] else "[bm25]"
    print(f"  {i+1}. {src:<8} {sim:<12} {doc['content'][:55]}...")

---
## 5. Junk Chunk Filtering — Why It Matters

The junk chunk filter was added to local-rag-llm after RAGAS evaluation revealed  
that Table of Contents chunks were consistently beating content chunks in retrieval.

**Why TOC chunks score high:**  
A TOC entry like "The 7-Pillar Agent Security Architecture...9" contains the exact section title.  
Any query about that section will match this TOC entry with very high similarity.  
The TOC chunk takes a retrieval slot that should go to the actual content.

**Impact on RAGAS scores (from real eval):**

| Run | Q1 context_recall | Cause |
|---|---|---|
| Without junk filter | 0.286 | TOC + Figure 1 caption took 2 of 4 slots |
| With junk filter | 0.857 | Content chunks fill the slots instead |

**The two junk conditions:**
1. Chunk contains "table of contents" — catches TOC pages
2. Chunk has ≥6 lines where >70% are short (<60 chars) AND total length <1500 chars — catches header lists

In [ ]:
# Testing the junk filter on real examples

test_chunks = [
    (
        "TOC page",
        "Table of contents\nIntroduction 6\nSecurity: The Evolution 7\n"
        "The Foundation: The 7-Pillar 9\nSandboxes and Supply Chain 13\n"
        "Ephemeral Sandboxing 13\nMitigating Hallucinated Packages 14"
    ),
    (
        "Figure caption (junk)",
        "Figure 1: The Secure Vibe Coding Agent Framework.\n"
        "This layered architecture differentiates the foundational\n"
        "security controls from high-velocity intent-driven defences.\n"
        "The following sections will deconstruct this architecture.\n"
        "Beginning with the baseline security harness.\n"
        "May 2026 9"
    ),
    (
        "Real content (keep)",
        "Pillar 1 - Infrastructure & Networking: Cloud Infrastructure Engineers must "
        "secure the foundational environment against upstream poisoning and container escapes. "
        "Because the harness must dictate where the agent's code actually runs and what it "
        "cannot reach, we isolate runtime execution within ephemeral, kernel-level sandboxes "
        "such as gVisor. Furthermore, strict network egress governance guarantees that "
        "agent-generated data travels only through authorised, offline caches."
    ),
    (
        "Short intro paragraph (keep)",
        "Software engineering is undergoing its most significant transformation since the "
        "introduction of high-level programming languages. The most profound shift is the "
        "transition from writing code to expressing intent."
    ),
]

print("JUNK CHUNK FILTER RESULTS")
print("=" * 60)
for label, content in test_chunks:
    junk = is_junk_chunk(content)
    status = "❌ REMOVED (junk)" if junk else "✅ KEPT (content)"
    print(f"\n{label}: {status}")
    print(f"  Length: {len(content)} chars")
    lines = [l.strip() for l in content.split('\n') if l.strip()]
    if lines:
        short = sum(1 for l in lines if len(l) < 60)
        print(f"  Lines: {len(lines)}, short (<60 chars): {short} ({short/len(lines)*100:.0f}%)")

---
## 6. Tuning the Retrieval Pipeline — What to Adjust and When

The retrieval pipeline has several parameters. Each has a specific signal to watch.

In [ ]:
# Parameter tuning guide

parameters = [
    {
        "parameter": "FAISS k (candidates)",
        "current": "k=16",
        "ragas_signal": "context_recall low on factual questions",
        "action": "Increase k — more candidates for CrossEncoder to choose from",
        "risk": "More noise in candidate pool, CrossEncoder works harder",
    },
    {
        "parameter": "Similarity threshold",
        "current": "0.3",
        "ragas_signal": "Irrelevant chunks in retrieved_contexts (low precision)",
        "action": "Raise threshold — filter more aggressively",
        "risk": "May drop valid chunks on niche/rare topics",
    },
    {
        "parameter": "CrossEncoder top_k",
        "current": "top_k=5",
        "ragas_signal": "context_precision low (noise in final chunks)",
        "action": "Reduce to 4 — tighter filter",
        "risk": "May drop a needed chunk for multi-part answers",
    },
    {
        "parameter": "BM25 / FAISS weights",
        "current": "40/60",
        "ragas_signal": "Exact ID/code queries failing (low recall on those Qs)",
        "action": "Raise BM25 weight to 50/50 or 60/40",
        "risk": "Semantic queries may degrade slightly",
    },
    {
        "parameter": "chunk_size",
        "current": "2000",
        "ragas_signal": "context_recall low on multi-point enumeration Qs",
        "action": "Increase to 2500 — fit more content per chunk",
        "risk": "Chunks less focused → CrossEncoder has harder time ranking",
    },
    {
        "parameter": "Junk filter thresholds",
        "current": "70% short lines, <1500 chars",
        "ragas_signal": "TOC/header chunks appearing in retrieved_contexts",
        "action": "Lower short-line threshold to 60% or increase char limit",
        "risk": "May remove legitimate short-paragraph content",
    },
]

print("PARAMETER TUNING GUIDE")
print("=" * 70)
for p in parameters:
    print(f"\n{p['parameter']} (current: {p['current']})")
    print(f"  Signal:  {p['ragas_signal']}")
    print(f"  Action:  {p['action']}")
    print(f"  Risk:    {p['risk']}")

print()
print("The correct tuning process:")
print("  1. Run RAGAS evaluation — get baseline metrics")
print("  2. Identify which questions have low context_recall or low context_precision")
print("  3. Inspect retrieved_contexts column in CSV — what's wrong?")
print("  4. Change ONE parameter")
print("  5. Re-run RAGAS — did that metric improve without hurting others?")
print("  6. Repeat until satisfied")

---
## 7. RRF vs Weighted Fusion — When to Use Each

In [ ]:
# Comparison: pure RRF vs weighted RRF vs score normalisation

print("FUSION METHOD COMPARISON")
print("=" * 70)

methods = [
    {
        "method": "Raw score addition",
        "used_in": "Naive implementations",
        "pros": "Simple",
        "cons": "Scale mismatch — one retriever dominates",
        "verdict": "❌ Never use",
    },
    {
        "method": "Score normalisation + weighted sum",
        "used_in": "Some production systems",
        "pros": "Preserves score information",
        "cons": "Normalisation is corpus-dependent, fragile across batches",
        "verdict": "⚠️ Use with caution",
    },
    {
        "method": "Pure RRF (equal weights)",
        "used_in": "Academic IR, some production",
        "pros": "Scale-independent, robust, no hyperparameter beyond k",
        "cons": "Cannot express domain knowledge (e.g. 'BM25 more important here')",
        "verdict": "✅ Good default",
    },
    {
        "method": "Weighted RRF (LangChain EnsembleRetriever)",
        "used_in": "local-rag-llm, LangChain apps",
        "pros": "Scale-independent + allows domain weighting",
        "cons": "Weights need tuning — default 40/60 may not be optimal",
        "verdict": "✅ Best for production RAG",
    },
]

for m in methods:
    print(f"\n{m['verdict']} {m['method']}")
    print(f"   Used in: {m['used_in']}")
    print(f"   Pros:    {m['pros']}")
    print(f"   Cons:    {m['cons']}")

---
## 8. Summary

| Concept | Key point |
|---|---|
| **Score incompatibility** | BM25 and cosine scores are on different scales — never add them directly |
| **RRF** | Rank-based fusion — completely scale-independent, robust across retriever types |
| **Weighted RRF** | LangChain EnsembleRetriever — RRF with per-retriever importance weights |
| **60/40 weighting** | FAISS 60%, BM25 40% — semantic matters more, but exact match is critical |
| **Deduplication** | Essential when FAISS and BM25 retrieve the same chunk |
| **Junk filter** | TOC and header chunks score high but contain no answer — filter before reranking |
| **Tuning order** | Start with RAGAS eval → identify failing questions → change one parameter → re-eval |
| **Pipeline order** | Query rewrite → FAISS (k=16, threshold) → BM25 (k=4) → merge+dedup → junk filter → CrossEncoder → top 5 |

**The hybrid retrieval insight:**  
The power of hybrid retrieval is not in the fusion algorithm — it's in having two complementary signals.  
Dense retrieval understands what the user means.  
Sparse retrieval knows exactly what the user typed.  
Together they cover the full spectrum of real queries.

---
## Next Notebooks

- **05** — Reranking with CrossEncoders (bi-encoder vs cross-encoder, ms-marco internals)
- **06** — RAG evaluation with RAGAS (metrics, golden dataset, reading results)
- **07** — RAG security (injection attacks, 3-layer defence in local-rag-llm)